In [1]:
from embedder import Embedder

embed = Embedder()

q1 = 'Can I still join the course after the start date?'
q2 = 'How to install Docker on Windows?'
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-05-20 14:58:51.361577554 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [4]:
v1.dot(dv)

np.float64(0.32332401928520804)

In [5]:
v2.dot(dv)

np.float64(0.019730453272608757)

In [6]:
# Сначала мы добавляем родительский каталог со вспомогательными скриптами в наш PYTHONPATH:
# Это позволяет нам импортировать ingest.py и rag_helper.py.
import sys
sys.path.insert(0, '..')

In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
texts = [doc['question'] + ' ' + doc['answer'] for doc in documents]

In [9]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/25 [00:00<?, ?it/s]

In [10]:
query = 'Can I still join the course after the start date?'
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [11]:
top5 = np.argsort(scores)[-5:] # indices of the top 5 most similar documents
top5 = top5[::-1] # reverse to get the most similar first
scores[top5] # scores of the top 5 most similar documents


array([0.76294095, 0.75793703, 0.7192132 , 0.65363119, 0.56010005])

In [12]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629409483375769
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.757937029277544
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.719213196115807
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section'

In [14]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

In [21]:
from minsearch import VectorSearch 

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents) # we build the index

In [18]:
from dotenv import load_dotenv
from ingest import load_faq_data, build_index
load_dotenv()

from openai import OpenAI
import os


model_name = "models/gemini-3.1-flash-lite"

client = OpenAI(
    api_key=os.getenv('GEMINI_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [19]:
from rag_helper import RAGBase, RAGVector

assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [20]:
query = 'I just discovered the course. Can I still join it?'
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [22]:
vector_assistant = RAGVector(
    embedder=embed,
    index=vindex,
    llm_client=client,
)

In [23]:
vector_assistant.rag('the program has already begun, can I still sign up?')

"Yes, you can still join. If you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework (while the form is open) even if you haven't registered."

In [24]:
print(vector_assistant.rag('which LLM model can be used'))

Based on the provided context, you can use the following LLM models:

*   **Groq:** `llama-3.3-70b-versatile`, DeepSeek R1, and Llama 4.
*   **Mistral:** Most models.
*   **Google Gemini:** `gemini-2.5-flash` and others.
*   **Ollama:** `llama3.1` and similar local models.

You can use any LLM platform for your experiments and project, though you may need to adjust your code to match the specific platform's documentation, tool schema, and response shape.
